In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from sklearn.model_selection import train_test_split

import config
from src.db_io import leer_tabla_sqlite

df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")
df = df[df["excluir_modelado"] == 0].reset_index(drop=True)

COLUMNAS_SENSIBLES = ["desc_genero", "grupo_edad", "desc_tipo_de_vivienda"]
COLUMNAS_FUGA = [c for c in df.columns if c.startswith("invesbot_") or c.startswith("inversion_virtual_")]
COLUMNAS_NO_FEATURE = ["numero_id", "etiqueta_adopcion", "excluir_modelado"] + COLUMNAS_SENSIBLES + COLUMNAS_FUGA

feature_cols = [c for c in df.columns if c not in COLUMNAS_NO_FEATURE]
X = pd.get_dummies(df[feature_cols], columns=["desc_segmento"], drop_first=True)
y = df["etiqueta_adopcion"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
)

# La mediana de estimador_ingreso se calcula SOLO sobre train y se aplica a train y test,
# para no filtrar estadísticas de test hacia el entrenamiento (leakage train/test).
mediana_estimador = X_train["estimador_ingreso"].median()
X_train["estimador_ingreso"] = X_train["estimador_ingreso"].fillna(mediana_estimador)
X_test["estimador_ingreso"] = X_test["estimador_ingreso"].fillna(mediana_estimador)

print(f"train: {X_train.shape}, test: {X_test.shape}, tasa adopción train: {y_train.mean():.4f}")

train: (615740, 36), test: (153935, 36), tasa adopción train: 0.0824


In [2]:
assert not any(c.startswith("invesbot_") or c.startswith("inversion_virtual_") for c in X.columns)
assert not any(c in X.columns for c in COLUMNAS_SENSIBLES)
print("OK: sin fuga de datos ni variables sensibles en las features")

OK: sin fuga de datos ni variables sensibles en las features


In [3]:
import json
import joblib
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score

# ~179 clientes en la población de modelado tienen NaN reales en 6 columnas
# financieras (ingresos_mensuales, total_egresos_mensuales, total_activos,
# total_pasivos, total_patrimonio, capacidad_ahorro) que Task 16 no imputó
# (solo cubrió estimador_ingreso). GradientBoostingClassifier no acepta NaN.
# Se imputa con la media de TRAIN únicamente (nunca de test), para no
# filtrar estadísticas de test hacia el entrenamiento.
media_train = X_train.mean()
X_train_filled = X_train.fillna(media_train)
X_test_filled = X_test.fillna(media_train)

modelo = GradientBoostingClassifier(random_state=config.RANDOM_STATE)
modelo.fit(X_train_filled, y_train)

proba = modelo.predict_proba(X_test_filled)[:, 1]
pred = modelo.predict(X_test_filled)

metricas = {
    "auc": roc_auc_score(y_test, proba),
    "precision": precision_score(y_test, pred),
    "recall": recall_score(y_test, pred),
}
print(metricas)

(config.OUTPUTS_DIR / "models").mkdir(parents=True, exist_ok=True)
joblib.dump(modelo, config.OUTPUTS_DIR / "models" / "propension_adopcion.pkl")
with open(config.OUTPUTS_DIR / "models" / "metricas_propension.json", "w") as f:
    json.dump(metricas, f, indent=2)

{'auc': 0.8718617190806525, 'precision': 0.5757188498402556, 'recall': 0.07106238662355076}


In [4]:
import numpy as np

# El AUC (0.87) mide calidad de ranking; el recall al umbral por defecto (0.5, 7%)
# es bajo porque la tasa base de adopción es ~7-8% y el modelo rara vez supera 0.5
# de probabilidad. Para un caso de uso de targeting de campaña, lo relevante no es
# el umbral fijo sino: "si contacto el top N% de clientes mejor puntuados, ¿qué
# recall/precisión obtengo?" — la métrica natural para priorizar contactos.
orden = np.argsort(-proba)
proba_ordenada = proba[orden]
y_ordenado = y_test.to_numpy()[orden]
n = len(y_test)

filas = []
for pct in [0.01, 0.05, 0.10, 0.20]:
    corte = max(1, int(np.ceil(n * pct)))
    umbral = proba_ordenada[corte - 1]
    seleccionados = y_ordenado[:corte]
    precision_topn = seleccionados.sum() / corte
    recall_topn = seleccionados.sum() / y_test.sum()
    filas.append({
        "top_pct": pct,
        "n_contactados": corte,
        "umbral_probabilidad": umbral,
        "precision": precision_topn,
        "recall": recall_topn,
    })

curva_top_n = pd.DataFrame(filas)
print(curva_top_n.to_string(index=False))

curva_top_n.to_csv(config.OUTPUTS_DIR / "models" / "curva_precision_recall.csv", index=False)

print(
    "\nInterpretación: el AUC alto (0.87) indica que el modelo ordena bien a los "
    "clientes por probabilidad de adopción, aun cuando el recall al umbral 0.5 sea "
    "bajo (7%). Para targeting de campaña -contactar el top N% de clientes con mayor "
    "score en vez de aplicar un umbral fijo de probabilidad- el modelo es útil: por "
    "ejemplo, contactando el 10% con mayor score se captura una fracción muy superior "
    "de adoptantes reales que un contacto aleatorio del mismo tamaño."
)

 top_pct  n_contactados  umbral_probabilidad  precision   recall
    0.01           1540             0.501500   0.577273 0.070116
    0.05           7697             0.364280   0.438093 0.265952
    0.10          15394             0.276112   0.375341 0.455714
    0.20          30787             0.155766   0.291974 0.708968

Interpretación: el AUC alto (0.87) indica que el modelo ordena bien a los clientes por probabilidad de adopción, aun cuando el recall al umbral 0.5 sea bajo (7%). Para targeting de campaña -contactar el top N% de clientes con mayor score en vez de aplicar un umbral fijo de probabilidad- el modelo es útil: por ejemplo, contactando el 10% con mayor score se captura una fracción muy superior de adoptantes reales que un contacto aleatorio del mismo tamaño.
